# 12 — 体系的最適化: 前処理 → 特徴選択 → アンサンブル

| 段階 | 目的 | 固定 |
|------|------|------|
| Stage 1 | 前処理の体系的最適化 | ET 全波数 |
| Stage 2 | 特徴選択の再確認 | Stage1 最良前処理 |
| Stage 3 | アンサンブルで分散低減 | Stage2 最良構成 |
| Stage 4 | 最終モデルで提出CSV作成 | |

**主指標 RMSE_le170**（実測≤170%のサンプルのみ、テスト域相当）。RMSE_all は参考値。  
**現行ベスト**: ET + sign≥0.8 + SNV+SG1(11,2)  RMSE_le170=14.20%

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import GroupKFold

from src.utils import load_data, parse_spectra, get_groups, make_submission, setup_japanese_font
from src.preprocessing import snv, savitzky_golay, msc as _msc_raw

plt.rcParams['figure.dpi'] = 110
setup_japanese_font()
SEED = 42

# ---- Data ----
train_df, test_df = load_data()
train_meta, y_s, X_raw, wn = parse_spectra(train_df)
test_meta, _, X_test_raw, _ = parse_spectra(test_df)
y = y_s.values.astype(float)
groups = get_groups(train_meta)
SPLITS = list(GroupKFold(n_splits=5).split(X_raw, y, groups))

# ---- Metrics ----
def rmse_all(yt, yp):
    return float(np.sqrt(np.mean((yt - yp)**2)))

def rmse_le(yt, yp, T=170.0):
    m = yt <= T
    return float(np.sqrt(np.mean((yt[m]-yp[m])**2))) if m.sum() > 0 else np.nan

# ---- Helpers ----
def detrend_spectra(X):
    idx = np.arange(X.shape[1]).astype(float)
    out = np.empty_like(X, dtype=float)
    for i, row in enumerate(X):
        p = np.polyfit(idx, row.astype(float), 1)
        out[i] = row - np.polyval(p, idx)
    return out

def corr_vec(X, yv):
    yc = yv - yv.mean(); Xc = X - X.mean(axis=0)
    num = (Xc * yc[:, None]).sum(0)
    den = np.sqrt((Xc**2).sum(0) * (yc**2).sum())
    with np.errstate(invalid='ignore', divide='ignore'):
        return np.where(den > 0, num/den, 0.0)

def make_sign_selector(sign_thresh, std_thresh):
    def selector(Xtr, ytr, gtr):
        r_list = [corr_vec(Xtr[gtr==sp], ytr[gtr==sp])
                  for sp in sorted(set(gtr)) if (gtr==sp).sum()>=3]
        r_mat = np.array(r_list)
        sc = np.abs(np.sign(r_mat).sum(0)) / len(r_list)
        return (sc >= sign_thresh) & (r_mat.std(0) <= std_thresh)
    return selector

# ---- Preprocessing factory ----
def make_preproc_factory(scale='snv', win=11, poly=2, deriv=1):
    """Returns factory(Xtr) -> preproc(X).  MSC fits reference on Xtr."""
    assert win % 2 == 1, f'window must be odd: {win}'
    assert win > poly, f'window ({win}) > polyorder ({poly}) required'
    def factory(Xtr):
        ref = Xtr.mean(axis=0) if scale == 'msc' else None
        def preproc(X):
            X = X.astype(float)
            if   scale == 'none':    Xs = X
            elif scale == 'snv':     Xs = snv(X)
            elif scale == 'msc':     Xs = _msc_raw(X, reference=ref)
            elif scale == 'detrend': Xs = detrend_spectra(X)
            else:                    raise ValueError(scale)
            return Xs if deriv == 0 else savitzky_golay(Xs, win, poly, deriv)
        return preproc
    return factory

# ---- CV runner (factory-based) ----
def run_cv2(preproc_factory, model_fn, feat_fn=None):
    fold_rows, oof_y_list, oof_p_list = [], [], []
    for fi, (tr, va) in enumerate(SPLITS):
        pp = preproc_factory(X_raw[tr])
        Xtr = pp(X_raw[tr]); Xva = pp(X_raw[va])
        ytr, yva = y[tr], y[va]; gtr = groups[tr]
        n_feat = Xtr.shape[1]
        if feat_fn is not None:
            sel = feat_fn(Xtr, ytr, gtr)
            Xtr = Xtr[:, sel]; Xva = Xva[:, sel]
            n_feat = int(sel.sum())
        m = model_fn(); m.fit(Xtr, ytr)
        pred = m.predict(Xva)
        fold_rows.append({'fold': fi+1, 'n_feat': n_feat,
                          'RMSE_all':   rmse_all(yva, pred),
                          'RMSE_le170': rmse_le(yva, pred)})
        oof_y_list.append(yva); oof_p_list.append(pred)
    oof_y = np.concatenate(oof_y_list)
    oof_p = np.concatenate(oof_p_list)
    return fold_rows, oof_y, oof_p

def summary_row(label, fold_rows, oof_y, oof_p):
    df = pd.DataFrame(fold_rows)
    means = df[['RMSE_all','RMSE_le170']].mean()
    return {'label': label,
            'n_feat': fold_rows[0]['n_feat'],
            'RMSE_all':   round(means['RMSE_all'],  2),
            'RMSE_le170': round(means['RMSE_le170'],2),
            'RMSE_le170_pooled': round(rmse_le(oof_y, oof_p), 2),
            'fold_le170': [round(r['RMSE_le170'],2) for r in fold_rows]}

print(f'Train: {X_raw.shape}  y: {y.min():.1f}-{y.max():.1f}%')
print(f'Samples with y<=170%: {(y<=170).sum()}/{len(y)} ({100*(y<=170).mean():.1f}%)')
CURRENT_BEST = 14.20  # RMSE_le170 baseline (notebook 11)

## Stage 1: 前処理の体系的最適化

ET(n=200, 全波数) 固定。前処理だけを動かす。  
Phase A: スケール×微分グリッド (win=11, poly=2固定)  
Phase B: SNV+SG1 と SNV+SG2 の window/poly スイープ

In [ ]:
# Stage 1 用モデル（高速スクリーニング）
ET_S1 = dict(n_estimators=200, max_features=0.3, random_state=SEED, n_jobs=-1)

# ---------- Phase A: scale x deriv grid (win=11, poly=2) ----------
PHASE_A = [
    ('raw',            make_preproc_factory('none',    11, 2, 0)),
    ('SNV',            make_preproc_factory('snv',     11, 2, 0)),
    ('MSC',            make_preproc_factory('msc',     11, 2, 0)),
    ('Detrend',        make_preproc_factory('detrend', 11, 2, 0)),
    ('SG1(11,2)',      make_preproc_factory('none',    11, 2, 1)),
    ('SG2(11,2)',      make_preproc_factory('none',    11, 2, 2)),
    ('SNV+SG1(11,2)',  make_preproc_factory('snv',     11, 2, 1)),  # current baseline
    ('SNV+SG2(11,2)',  make_preproc_factory('snv',     11, 2, 2)),
    ('MSC+SG1(11,2)',  make_preproc_factory('msc',     11, 2, 1)),
    ('MSC+SG2(11,2)',  make_preproc_factory('msc',     11, 2, 2)),
    ('Detrend+SG1(11,2)', make_preproc_factory('detrend', 11, 2, 1)),
]

# ---------- Phase B: window/poly sweep (SNV+SG1 and SNV+SG2) ----------
PHASE_B = []
for scale in ['snv', 'msc']:  # top-2 scale candidates from NIR domain knowledge
    for win in [7, 15, 21]:
        for poly in [2, 3]:
            if win > poly:
                for deriv in [1, 2]:
                    lbl = f'{scale.upper()}+SG{deriv}({win},{poly})'
                    PHASE_B.append((lbl, make_preproc_factory(scale, win, poly, deriv)))
    # also poly=3 with win=11
    for deriv in [1, 2]:
        lbl = f'{scale.upper()}+SG{deriv}(11,3)'
        PHASE_B.append((lbl, make_preproc_factory(scale, 11, 3, deriv)))

print(f'Phase A: {len(PHASE_A)} configs')
print(f'Phase B: {len(PHASE_B)} configs')
print(f'Total Stage 1: {len(PHASE_A)+len(PHASE_B)} configs')

In [ ]:
# ---------- Stage 1 実行 ----------
s1_results = []  # list of summary dicts
s1_oof = {}      # label -> (oof_y, oof_p)

all_s1_configs = PHASE_A + PHASE_B
print(f'Running {len(all_s1_configs)} Stage 1 configs (ET n=200)...')

for i, (lbl, pf) in enumerate(all_s1_configs):
    rows, oy, op = run_cv2(pf, lambda: ExtraTreesRegressor(**ET_S1))
    sr = summary_row(lbl, rows, oy, op)
    s1_results.append(sr)
    s1_oof[lbl] = (oy, op)
    marker = ' <-- current' if lbl == 'SNV+SG1(11,2)' else ''
    print(f'  [{i+1:2d}/{len(all_s1_configs)}] {lbl:30s} '
          f'RMSE_le170={sr["RMSE_le170"]:5.2f}%  RMSE_all={sr["RMSE_all"]:5.2f}%{marker}')

print('\nStage 1 done.')

In [ ]:
# ---------- Stage 1 結果分析 ----------
df_s1 = pd.DataFrame(s1_results).sort_values('RMSE_le170')

print('\n=== Stage 1 Results (sorted by RMSE_le170) ===')
print(f'{"Rank":>4} {"Label":>30} {"n_feat":>7} {"RMSE_le170":>11} {"RMSE_all":>10} {"Folds(le170)":>20}')
print('-' * 90)
baseline_row = df_s1[df_s1['label'] == 'SNV+SG1(11,2)'].iloc[0]
for rank, (_, r) in enumerate(df_s1.head(20).iterrows(), 1):
    delta = r['RMSE_le170'] - baseline_row['RMSE_le170']
    marker = ' [baseline]' if r['label'] == 'SNV+SG1(11,2)' else \
             (' [BETTER]' if delta < -0.2 else '')
    folds_str = str(r['fold_le170'])
    print(f'{rank:>4} {r["label"]:>30} {r["n_feat"]:>7} '
          f'{r["RMSE_le170"]:>11.2f} {r["RMSE_all"]:>10.2f} '
          f'{folds_str:>20}{marker}')

# Top 3 for Stage 2
top3_preprocs = list(df_s1.head(3)['label'].values)
print(f'\n>>> Stage 2 に進む上位3前処理:')
for i, lbl in enumerate(top3_preprocs, 1):
    row = df_s1[df_s1['label'] == lbl].iloc[0]
    print(f'  {i}. {lbl}: RMSE_le170={row["RMSE_le170"]:.2f}%')

# ---- 可視化: top 10 の比較 ----
fig, ax = plt.subplots(figsize=(12, 5))
top10 = df_s1.head(10)
x = np.arange(10)
w = 0.35
bars1 = ax.bar(x-w/2, top10['RMSE_le170'], w, label='RMSE_le170 (主)', color='darkorange', alpha=0.85)
bars2 = ax.bar(x+w/2, top10['RMSE_all'],   w, label='RMSE_all (参考)',  color='steelblue',  alpha=0.7)
ax.axhline(baseline_row['RMSE_le170'], color='darkorange', ls='--', lw=1.5,
           label=f'Baseline le170={baseline_row["RMSE_le170"]:.2f}%')
ax.set_xticks(x)
ax.set_xticklabels(top10['label'], rotation=35, ha='right', fontsize=8)
ax.set_ylabel('RMSE (%)')
ax.set_title('Stage 1 Top 10 前処理比較 (ET n=200, 全波数)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('../results/s1_preprocessing.png', dpi=150, bbox_inches='tight')
plt.show()

## Stage 2: 特徴選択の再確認

Stage 1 の上位3前処理に対し、全波数 vs 一貫性選択（複数閾値）を比較。  
モデルは ET(n=300) に戻す。前処理が変わると最適な選択閾値も変わりうる。

In [ ]:
ET_S2 = dict(n_estimators=300, max_features=0.3, random_state=SEED, n_jobs=-1)

# Top 3 preprocessing factories
top3_factories = {lbl: dict(all_s1_configs)[lbl] for lbl in top3_preprocs
                  if lbl in dict(all_s1_configs)}

# 特徴選択の組み合わせ
FEAT_CONFIGS = [
    ('all',             None),
    ('sign0.8/std0.05', make_sign_selector(0.8, 0.05)),  # current
    ('sign0.9/std0.05', make_sign_selector(0.9, 0.05)),
    ('sign1.0/std0.05', make_sign_selector(1.0, 0.05)),
    ('sign0.8/std0.10', make_sign_selector(0.8, 0.10)),
    ('sign0.9/std0.10', make_sign_selector(0.9, 0.10)),
]

s2_results = []
s2_oof = {}  # (preproc_lbl, feat_lbl) -> (oof_y, oof_p)

n_total = len(top3_preprocs) * len(FEAT_CONFIGS)
print(f'Running {n_total} Stage 2 configs (ET n=300)...')

idx = 0
for plbl in top3_preprocs:
    pf = top3_factories[plbl]
    for flbl, ffn in FEAT_CONFIGS:
        idx += 1
        lbl_full = f'{plbl} | {flbl}'
        rows, oy, op = run_cv2(pf, lambda: ExtraTreesRegressor(**ET_S2), ffn)
        sr = summary_row(lbl_full, rows, oy, op)
        s2_results.append(sr)
        s2_oof[(plbl, flbl)] = (oy, op)
        is_cur = (plbl == 'SNV+SG1(11,2)' and flbl == 'sign0.8/std0.05')
        marker = ' <-- current' if is_cur else ''
        print(f'  [{idx:2d}/{n_total}] {lbl_full:50s} '
              f'le170={sr["RMSE_le170"]:5.2f}%  all={sr["RMSE_all"]:5.2f}%{marker}')

print('\nStage 2 done.')

In [ ]:
# ---------- Stage 2 結果分析 ----------
df_s2 = pd.DataFrame(s2_results).sort_values('RMSE_le170')

print('\n=== Stage 2 Results (sorted by RMSE_le170) ===')
cur_mask = (df_s2['label'].str.startswith('SNV+SG1(11,2)') &
             df_s2['label'].str.endswith('sign0.8/std0.05'))
if cur_mask.any():
    current_le170_s2 = df_s2[cur_mask].iloc[0]['RMSE_le170']
else:
    current_le170_s2 = CURRENT_BEST

print(f'{"Rank":>4} {"Label":>52} {"n_feat":>7} {"RMSE_le170":>11} {"RMSE_all":>10}')
print('-' * 92)
for rank, (_, r) in enumerate(df_s2.iterrows(), 1):
    delta = r['RMSE_le170'] - current_le170_s2
    marker = (' [current]' if 'SNV+SG1(11,2)' in r['label'] and 'sign0.8/std0.05' in r['label']
              else (' [BETTER]' if delta < -0.2 else ''))
    print(f'{rank:>4} {r["label"]:>52} {r["n_feat"]:>7} '
          f'{r["RMSE_le170"]:>11.2f} {r["RMSE_all"]:>10.2f}{marker}')

# Best Stage 2 config
best_s2 = df_s2.iloc[0]
best_s2_plbl = best_s2['label'].split(' | ')[0]
best_s2_flbl = best_s2['label'].split(' | ')[1]
best_s2_pf   = top3_factories[best_s2_plbl]
best_s2_ffn  = dict(FEAT_CONFIGS)[best_s2_flbl]
best_s2_oof_y, best_s2_oof_p = s2_oof[(best_s2_plbl, best_s2_flbl)]

print(f'\n>>> Stage 3 に持ち込む最良構成:')
print(f'    前処理: {best_s2_plbl}')
print(f'    特徴:   {best_s2_flbl}')
print(f'    RMSE_le170 = {best_s2["RMSE_le170"]:.2f}%  (現行比 '
      f'{best_s2["RMSE_le170"]-current_le170_s2:+.2f}%)')

## Stage 3: アンサンブルで分散低減

Stage 2 最良構成をベースに 4 種のアンサンブルを比較する。  
アンサンブル重みの最適化は **全OOFで実施**（テストセットは一切使わない）。

In [ ]:
ET_S3 = dict(n_estimators=300, max_features=0.3, n_jobs=-1)  # seed は後で指定

# ---- (A) シード平均 ----
print('Stage 3A: Seed ensemble (5 seeds)...')
SEEDS = [42, 0, 1, 2, 3]
seed_oof_preds = []
for seed in SEEDS:
    _, _, op = run_cv2(best_s2_pf,
                       lambda s=seed: ExtraTreesRegressor(**ET_S3, random_state=s),
                       best_s2_ffn)
    seed_oof_preds.append(op)

oof_seed_avg  = np.mean(seed_oof_preds, axis=0)
oof_ref_y     = best_s2_oof_y   # fold assignment is same for all
seed_le170    = rmse_le(oof_ref_y, oof_seed_avg)
seed_all      = rmse_all(oof_ref_y, oof_seed_avg)
print(f'  Seed avg: RMSE_le170={seed_le170:.2f}%  RMSE_all={seed_all:.2f}%')
print(f'  Single best (Stage2):  RMSE_le170={best_s2["RMSE_le170"]:.2f}%')

In [ ]:
# ---- (B) 前処理多様性アンサンブル ----
# Top 3 preprocessing の OOF 予測を平均
# これらは Stage 2 で各前処理の 'best feature selection' で計算済み
print('\nStage 3B: Preprocessing diversity ensemble (top 3 preprocessings)...')

# top 3 preprocessing の最良特徴選択でのOOFを収集
preproc_div_preds = []
preproc_div_labels = []
for plbl in top3_preprocs:
    # 各前処理の中で best feature selection を見つける
    sub = df_s2[df_s2['label'].str.startswith(plbl + ' | ')]
    best_flbl = sub.sort_values('RMSE_le170').iloc[0]['label'].split(' | ')[1]
    _, op = s2_oof[(plbl, best_flbl)]
    preproc_div_preds.append(op)
    preproc_div_labels.append(f'{plbl}|{best_flbl}')

oof_preproc_avg = np.mean(preproc_div_preds, axis=0)
preproc_le170 = rmse_le(oof_ref_y, oof_preproc_avg)
preproc_all   = rmse_all(oof_ref_y, oof_preproc_avg)
print(f'  Preprocessing diversity avg: RMSE_le170={preproc_le170:.2f}%  RMSE_all={preproc_all:.2f}%')
print(f'  Components: {preproc_div_labels}')

In [ ]:
# ---- (C) モデル多様性アンサンブル ----
# 最良前処理×最良特徴選択で ET, RF, HistGB の OOF を取得し平均
print('\nStage 3C: Model diversity ensemble (ET + RF + HistGB)...')

RF_S3  = dict(n_estimators=300, max_features=0.3, random_state=SEED, n_jobs=-1)

_, _, oof_rf  = run_cv2(best_s2_pf,
                        lambda: RandomForestRegressor(**RF_S3), best_s2_ffn)
_, _, oof_hgb = run_cv2(best_s2_pf,
                        lambda: HistGradientBoostingRegressor(max_iter=300, random_state=SEED),
                        best_s2_ffn)

model_preds = [best_s2_oof_p, oof_rf, oof_hgb]
model_names = ['ET', 'RF', 'HistGB']
for name, op in zip(model_names, model_preds):
    print(f'  {name:6s}: RMSE_le170={rmse_le(oof_ref_y, op):.2f}%  RMSE_all={rmse_all(oof_ref_y, op):.2f}%')

oof_model_avg = np.mean(model_preds, axis=0)
model_le170 = rmse_le(oof_ref_y, oof_model_avg)
model_all   = rmse_all(oof_ref_y, oof_model_avg)
print(f'  Equal avg:  RMSE_le170={model_le170:.2f}%  RMSE_all={model_all:.2f}%')

In [ ]:
# ---- (D) OOF最適重みアンサンブル ----
# ET×seed + 前処理多様性 + モデル多様性 を1つに統合。OOFで重みを最適化。
print('\nStage 3D: OOF-optimized weighted ensemble...')

# 候補OOF予測の全リスト
cand_preds  = seed_oof_preds + preproc_div_preds + [oof_rf, oof_hgb]
cand_labels = ([f'ET_seed{s}' for s in SEEDS] +
               preproc_div_labels + ['RF_best', 'HistGB_best'])

n_cand = len(cand_preds)

def neg_rmse_le(w):
    w_abs = np.abs(w); w_n = w_abs / w_abs.sum()
    oof = sum(ww * pp for ww, pp in zip(w_n, cand_preds))
    return rmse_le(oof_ref_y, oof)

res = minimize(neg_rmse_le, np.ones(n_cand)/n_cand,
               method='Nelder-Mead',
               options={'maxiter': 5000, 'xatol': 1e-5, 'fatol': 1e-5})
w_opt = np.abs(res.x); w_opt /= w_opt.sum()

oof_weighted = sum(w * p for w, p in zip(w_opt, cand_preds))
wtd_le170 = rmse_le(oof_ref_y, oof_weighted)
wtd_all   = rmse_all(oof_ref_y, oof_weighted)
print(f'  Weighted: RMSE_le170={wtd_le170:.2f}%  RMSE_all={wtd_all:.2f}%')
print('\n  OOF weights (non-zero):')
for lbl, w in zip(cand_labels, w_opt):
    if w > 0.01:
        print(f'    {lbl:30s}: {w:.3f}')

In [ ]:
# ---- Stage 3 比較サマリー ----
s3_summary = [
    {'label': 'Single best (Stage2)',       'RMSE_le170': best_s2['RMSE_le170'],
                                             'RMSE_all':   best_s2['RMSE_all']},
    {'label': 'Seed avg (5 seeds)',          'RMSE_le170': seed_le170,
                                             'RMSE_all': seed_all},
    {'label': 'Preprocessing diversity avg', 'RMSE_le170': preproc_le170,
                                             'RMSE_all': preproc_all},
    {'label': 'Model diversity avg (ET+RF+HGB)', 'RMSE_le170': model_le170,
                                             'RMSE_all': model_all},
    {'label': 'OOF-optimized weighted',     'RMSE_le170': wtd_le170,
                                             'RMSE_all': wtd_all},
]
df_s3 = pd.DataFrame(s3_summary).sort_values('RMSE_le170')

print('\n=== Stage 3 アンサンブル比較 ===')
print(f'{"Label":>38} {"RMSE_le170":>12} {"RMSE_all":>10} {"vs_current":>12}')
print('-' * 78)
for _, r in df_s3.iterrows():
    delta = r['RMSE_le170'] - current_le170_s2
    mark = ' [BETTER]' if delta < -0.2 else ('  [tie]' if abs(delta) <= 0.2 else '')
    print(f'{r["label"]:>38} {r["RMSE_le170"]:>12.2f} {r["RMSE_all"]:>10.2f} '
          f'{delta:>+11.2f}%{mark}')

# 最良アンサンブル構成を決定
best_s3 = df_s3.iloc[0]
best_s3_label = best_s3['label']
print(f'\n>>> Stage 4 採用: {best_s3_label}')
print(f'    RMSE_le170 = {best_s3["RMSE_le170"]:.2f}%')

# 可視化
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(df_s3))
ax.bar(x, df_s3['RMSE_le170'], color='darkorange', alpha=0.85, label='RMSE_le170 (主)')
ax.bar(x, df_s3['RMSE_all'], alpha=0.45, color='steelblue', label='RMSE_all (参考)')
ax.axhline(current_le170_s2, color='red', ls='--', lw=1.5,
           label=f'current={current_le170_s2:.2f}%')
ax.set_xticks(x)
ax.set_xticklabels(df_s3['label'], rotation=20, ha='right', fontsize=9)
ax.set_ylabel('RMSE (%)')
ax.set_title('Stage 3 アンサンブル比較')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('../results/s3_ensemble.png', dpi=150, bbox_inches='tight')
plt.show()

## Stage 4: 最終モデルで提出CSV作成

Stage 3 の最良構成を全訓練データで学習 → test を予測 → T=200 クリップ → CSV 保存。

In [ ]:
print(f'=== Stage 4 最終提出 ===')
print(f'採用構成: {best_s3_label}')
print(f'前処理:   {best_s2_plbl}')
print(f'特徴選択: {best_s2_flbl}')

# 全訓練データで前処理 (MSCはXtr_all=全訓練データでfit)
pp_final = best_s2_pf(X_raw)  # fit on all training
X_tr_pp  = pp_final(X_raw)
X_te_pp  = pp_final(X_test_raw)

# 特徴選択 (全訓練データのみで計算)
if best_s2_ffn is not None:
    sel_final = best_s2_ffn(X_tr_pp, y, groups)
    X_tr_sel = X_tr_pp[:, sel_final]
    X_te_sel = X_te_pp[:, sel_final]
    print(f'  選択波数: {sel_final.sum()} / {len(sel_final)}')
else:
    X_tr_sel = X_tr_pp
    X_te_sel = X_te_pp
    print(f'  特徴選択なし (全波数: {X_tr_sel.shape[1]})')

# ---- 最良アンサンブル構成に応じて予測 ----
if 'Seed' in best_s3_label:
    print('  -> Seed ensemble (5 seeds, all training data)')
    test_preds = []
    for seed in SEEDS:
        m = ExtraTreesRegressor(**ET_S3, random_state=seed)
        m.fit(X_tr_sel, y)
        test_preds.append(m.predict(X_te_sel))
    final_pred = np.mean(test_preds, axis=0)
elif 'Preprocessing' in best_s3_label:
    print('  -> Preprocessing diversity ensemble (top 3 preprocessings)')
    test_preds = []
    for plbl in top3_preprocs:
        pf = top3_factories[plbl]
        sub = df_s2[df_s2['label'].str.startswith(plbl + ' | ')]
        best_flbl_div = sub.sort_values('RMSE_le170').iloc[0]['label'].split(' | ')[1]
        ffn_div = dict(FEAT_CONFIGS)[best_flbl_div]
        pp_div = pf(X_raw)  # fit on all training
        Xtr_d = pp_div(X_raw); Xte_d = pp_div(X_test_raw)
        if ffn_div is not None:
            sel_d = ffn_div(Xtr_d, y, groups)
            Xtr_d = Xtr_d[:, sel_d]; Xte_d = Xte_d[:, sel_d]
        m = ExtraTreesRegressor(**ET_S3, random_state=SEED)
        m.fit(Xtr_d, y)
        test_preds.append(m.predict(Xte_d))
    final_pred = np.mean(test_preds, axis=0)
elif 'weighted' in best_s3_label.lower():
    print('  -> Weighted ensemble (retraining all components on all training data)')
    test_preds_for_weighted = []
    for seed in SEEDS:
        m = ExtraTreesRegressor(**ET_S3, random_state=seed)
        m.fit(X_tr_sel, y)
        test_preds_for_weighted.append(m.predict(X_te_sel))
    for plbl in top3_preprocs:
        pf = top3_factories[plbl]
        sub = df_s2[df_s2['label'].str.startswith(plbl + ' | ')]
        best_flbl_div = sub.sort_values('RMSE_le170').iloc[0]['label'].split(' | ')[1]
        ffn_div = dict(FEAT_CONFIGS)[best_flbl_div]
        pp_div = pf(X_raw)
        Xtr_d = pp_div(X_raw); Xte_d = pp_div(X_test_raw)
        if ffn_div is not None:
            sel_d = ffn_div(Xtr_d, y, groups)
            Xtr_d = Xtr_d[:, sel_d]; Xte_d = Xte_d[:, sel_d]
        m = ExtraTreesRegressor(**ET_S3, random_state=SEED)
        m.fit(Xtr_d, y)
        test_preds_for_weighted.append(m.predict(Xte_d))
    # RF and HistGB
    m_rf = RandomForestRegressor(**RF_S3); m_rf.fit(X_tr_sel, y)
    m_hgb = HistGradientBoostingRegressor(max_iter=300, random_state=SEED); m_hgb.fit(X_tr_sel, y)
    test_preds_for_weighted.append(m_rf.predict(X_te_sel))
    test_preds_for_weighted.append(m_hgb.predict(X_te_sel))
    final_pred = sum(w * p for w, p in zip(w_opt, test_preds_for_weighted))
else:
    print('  -> Single best ET model')
    m = ExtraTreesRegressor(**ET_S3, random_state=SEED)
    m.fit(X_tr_sel, y)
    final_pred = m.predict(X_te_sel)

# クリップ (T=200)
final_pred_clipped = np.clip(final_pred, 0, 200)
print(f'  Test predictions: min={final_pred_clipped.min():.1f}  '
      f'max={final_pred_clipped.max():.1f}  mean={final_pred_clipped.mean():.1f}%')
print(f'  Clipped at 200%: {(final_pred > 200).sum()} samples')

# 提出ファイル名に構成を含める
tag = best_s3_label.lower().replace(' ', '_').replace('(', '').replace(')', '').replace('+', 'p')[:25]
sub_path = f'../submissions/s4_{tag}.csv'
sub = make_submission(test_meta, final_pred_clipped, path=sub_path)
print(f'  Saved: {sub_path}  ({len(sub)} rows)')

## 総括

In [ ]:
print('=' * 72)
print('体系的最適化 総括')
print('=' * 72)

print(f'\n[到達した RMSE_le170]')
print(f'  現行ベスト (notebook11):  {CURRENT_BEST:.2f}%')
print(f'  Stage2 最良単一モデル:    {best_s2["RMSE_le170"]:.2f}%  ({best_s2["RMSE_le170"]-CURRENT_BEST:+.2f}%)')
print(f'  Stage3 最良アンサンブル:  {best_s3["RMSE_le170"]:.2f}%  ({best_s3["RMSE_le170"]-CURRENT_BEST:+.2f}%)')

print(f'\n[効いた要素 (Stage1)]')
s1_top1 = df_s1.iloc[0]
if s1_top1['label'] == 'SNV+SG1(11,2)':
    print(f'  前処理: 現行(SNV+SG1 win11)が最良または同等。変更不要。')
else:
    print(f'  前処理: {s1_top1["label"]} が改善 ({s1_top1["RMSE_le170"]-baseline_row["RMSE_le170"]:+.2f}%)')

print(f'\n[効いた要素 (Stage2: 特徴選択)]')
best_feat_desc = best_s2['label'].split(' | ')[1]
if best_feat_desc == 'sign0.8/std0.05':
    print(f'  特徴選択: 現行(sign0.8/std0.05)が最良。変更不要。')
else:
    print(f'  特徴選択: {best_feat_desc} が改善')

print(f'\n[効いた要素 (Stage3: アンサンブル)]')
s3_best = df_s3.iloc[0]
s3_second = df_s3.iloc[1]
delta_s3 = s3_best['RMSE_le170'] - current_le170_s2
if delta_s3 < -0.2:
    print(f'  アンサンブル: {s3_best["label"]} が {delta_s3:+.2f}% 改善')
else:
    print(f'  アンサンブル: 最良={s3_best["label"]} ({delta_s3:+.2f}%) -- 誤差範囲の可能性あり')

print(f'\n[まだ詰められそうな方向 (10%切りに向けて)]')
print('  - 特徴設計: 4760帯の局所的特徴 (絶対値/比率) を数値特徴として追加')
print('  - 前処理: window/polyorder の細かいグリッドサーチ (有望な組み合わせを絞り込んで)')
print('  - 樹種間汎化: テスト樹種に近い訓練樹種にサンプル重みを増やす (sample_weight)')
print('  - Fold3除外: ベイスギ(sp15)の高含水域を学習から除いた際の影響を試す (注意: 情報損失)')
print(f'\n提出ファイル: {sub_path}')